# PEFTGuard retrained on the shared split (Sandra)

Colab Pro, **A100** runtime. Runs the repo's `shared_protocol.detectors.peftguard` code; nothing here re-implements it.

Order matters and follows the protocol:
1. fit on the shared **train** split, early-stop on the shared **validation** split (cells 1-7)
2. freeze: score the **calibration** adapters, save the clean-only 5% conformal threshold (cell 8)
3. optional reference line: the authors' pretrained checkpoint (cell 9)
4. evaluation (cell 10) stays **locked** until every detector (spectral, PEFTGuard, Z-PEFT, one-class) is frozen.

Never edit the threshold, the split or the seed after seeing evaluation numbers.

## 1. GPU, dependencies, memory plan

In [ ]:
!pip -q install safetensors
import torch
gpu = torch.cuda.get_device_properties(0)
total_gb = gpu.total_memory / 2**30
print(gpu.name, f"{total_gb:.0f} GB")
assert total_gb >= 26, "PEFTGuard Llama-2-7B needs a >=26 GB GPU (A100-40GB recommended); an L4 (24 GB) does not fit. Runtime > Change runtime type"

## 2. Configuration (edit only this cell)
`MANIFEST` and `SPLIT` are Julia's verified manifest and saved split. `ADAPTER_ROOT` is local disk (fast); the finished run is copied to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/JuliaIssa2/FYP-AI-Antivirus.git"   # <- the team repo
REPO_BRANCH = "main"
MANIFEST   = "/content/drive/MyDrive/fyp_shared/manifest.json"
SPLIT      = "/content/drive/MyDrive/fyp_shared/splits.json"
ADAPTER_ROOT = "/content/adapters"                              # <root>/<adapter_id>/adapter_model.safetensors
RUN_DIR    = "/content/runs/peftguard_seed0"                    # local disk, written once, never overwritten
DRIVE_OUT  = "/content/drive/MyDrive/fyp_shared/peftguard_seed0"
SEED = 0
PLAN = "auto"   # auto | adafactor_fp32 | adam_bf16 | adam_fp32  (auto picks by GPU memory, see the next section)

## 3. Get the code

In [ ]:
import os, sys
if not os.path.isdir("/content/repo"):
    !git clone --branch $REPO_BRANCH $REPO_URL /content/repo
%cd /content/repo
sys.path.insert(0, "/content/repo")
!git log -1 --format='%H %s'

## 4. Load the verified manifest and fixed split

In [ ]:
from shared_protocol.manifest import load_manifest
from shared_protocol.splits import load_split, development_records
from shared_protocol.scoring import bind_protocol

manifest = load_manifest(MANIFEST, require_verified=True)   # refuses unverified metadata
plan = load_split(SPLIT, manifest)                          # recomputes and checks the whole assignment
binding = bind_protocol(manifest, plan)
for part in ("train", "validation", "calibration", "evaluation", "final_holdout"):
    print(f"{part:14s} {len(binding.ids(part))} adapters")
train_records = development_records(manifest, plan, "train")
val_records = development_records(manifest, plan, "validation")
cal_records = development_records(manifest, plan, "calibration")

## 5. Stage adapter files
Only the partitions being used are staged. HF-hosted adapters (`hf://datasets/<repo>@<rev>/<dir>/adapter_model.safetensors`) are downloaded;
for any other locator copy the file to `ADAPTER_ROOT/<adapter_id>/adapter_model.safetensors` yourself.
Training and scoring re-hash every file against the manifest SHA256, so a wrong or partial download fails loudly.
Add `HF_TOKEN` in Colab *Secrets* (key icon) for the private repo.

In [ ]:
import shutil, re
from pathlib import Path
from huggingface_hub import hf_hub_download
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

def stage(records):
    for row in records:
        dest = Path(ADAPTER_ROOT) / row["adapter_id"] / "adapter_model.safetensors"
        if dest.exists():
            continue
        m = re.fullmatch(r"hf://datasets/([^@]+)@([^/]+)/(.+)", row["path_or_source"])
        if not m:
            raise FileNotFoundError(f"{row['adapter_id']}: not an hf:// locator; stage it manually to {dest}")
        repo, rev, path = m.groups()
        src = hf_hub_download(repo, path, repo_type="dataset", revision=rev, token=HF_TOKEN)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dest)

stage(train_records); stage(val_records); stage(cal_records)
print("staged", sum(1 for _ in Path(ADAPTER_ROOT).glob("*/adapter_model.safetensors")), "adapters")

## 6. Train (retrain PEFTGuard on exactly the shared train split)
Memory plans, MEASURED on an A100-40GB (peak GPU memory): `adam_fp32` (the authors' recipe) runs out of memory and needs a >=70 GB GPU;
`adafactor_fp32` (fp32 weights, Adafactor) peaks at 28.1 GB; `adam_bf16` (everything bf16) at 22.0 GB. 8-bit Adam is NOT used: it crashes on the
2^31-element `fc1` weight. `auto` picks `adafactor_fp32` on a 40 GB A100. Any plan other than `adam_fp32` is a deviation: state it with the results.
The plan is written into `fit.json`. Run a 1-epoch smoke test first (`EPOCHS = 1`, a different `RUN_DIR`) and check the loss is finite and falling.
For a cross-check run, set `PLAN = "adam_bf16"` with its own `RUN_DIR`/`DRIVE_OUT`.

In [ ]:
from shared_protocol.detectors.peftguard.train import TrainConfig, PLANS, auto_memory_plan, fit
from shared_protocol.detectors.peftguard.converter import root_resolver

EPOCHS = 20
optimizer, param_dtype = auto_memory_plan(total_gb) if PLAN == "auto" else PLANS[PLAN]
cfg = TrainConfig(epochs=EPOCHS, seed=SEED, optimizer=optimizer, param_dtype=param_dtype)
print(cfg)
record = fit(binding, train_records, val_records, cfg, RUN_DIR, resolver=root_resolver(ADAPTER_ROOT))
print("checkpoint sha256:", record["checkpoint_sha256"])

## 7. Keep the run (Drive). Write the checkpoint SHA256 into the team log: it is the detector's frozen state.

In [ ]:
os.makedirs(DRIVE_OUT, exist_ok=False)
for name in ("peftguard_best.pt", "fit.json"):
    shutil.copyfile(f"{RUN_DIR}/{name}", f"{DRIVE_OUT}/{name}")
print(open(f"{DRIVE_OUT}/fit.json").read()[:300], "...")

## 8. Freeze: clean-only calibration threshold (5% target)
Uses only the calibration partition (one representative per leakage group). No evaluation data is touched.

In [ ]:
from shared_protocol.detectors.peftguard.detector import PEFTGuardDetector
from shared_protocol.scoring import score_development
from shared_protocol.calibration import calibrate_benign, save_threshold

detector = PEFTGuardDetector.from_fit(f"{RUN_DIR}/fit.json", f"{RUN_DIR}/peftguard_best.pt", binding,
                                      expected_checkpoint_sha256=record["checkpoint_sha256"],
                                      resolver=root_resolver(ADAPTER_ROOT))
cal_scores = score_development(detector, cal_records, binding, "calibration")
threshold = calibrate_benign(binding, cal_records, cal_scores)
print("n calibration:", len(cal_scores.scores), "| threshold:", threshold.value)
import statistics
raws = sorted(s.raw for s in cal_scores.scores)
print("raw logit-difference range:", raws[0], "to", raws[-1], "| distinct:", len(set(raws)))
save_threshold(f"{DRIVE_OUT}/threshold.json", threshold, binding)

## 9. (Optional) Reference line: the authors' pretrained Llama-2-7B checkpoint
Reference only. Its training adapters are unknown (question 1 to Zhen Sun), so it is never the baseline. Needs ~13 GB + 8.6 GB download.

In [ ]:
from shared_protocol.detectors.peftguard.converter import sha256_file
CKPT_FILE = "toxic-backdoors-hard/llama2_7b_toxic_backdoors_hard1/best_model.pth"
ckpt = hf_hub_download("Vincent-HKUSTGZ/PEFTGuard_For_Llama2_7B", CKPT_FILE, token=HF_TOKEN)
ckpt_sha = sha256_file(ckpt); print("pretrained sha256:", ckpt_sha)   # record this in the team log
ref = PEFTGuardDetector.pretrained_reference(
    ckpt, expected_sha256=ckpt_sha, resolver=root_resolver(ADAPTER_ROOT),
    provenance="Authors' released checkpoint " + CKPT_FILE + "; training adapters unknown (author not yet answered)")
ref_cal = score_development(ref, cal_records, binding, "calibration")
print("reference calibration threshold:", calibrate_benign(binding, cal_records, ref_cal).value)

## 10. Evaluation: LOCKED
Run only after the whole team has frozen all detectors and thresholds and the custodian (Julia) has released the evaluation view.
This cell never fits, tunes or re-calibrates anything.

In [ ]:
RELEASE_EVALUATION = False        # <- the custodian flips this; nobody changes it to "see how it looks"
assert RELEASE_EVALUATION, "Evaluation is locked until all detectors are frozen"

from copy import deepcopy
from shared_protocol.evaluation import score_evaluation, evaluate, save_evaluation
eval_ids = set(binding.ids("evaluation"))
eval_records = [deepcopy(r) for r in manifest["adapters"] if r["adapter_id"] in eval_ids]
stage(eval_records)
scored = score_evaluation(detector, eval_records, binding, threshold)
save_evaluation(f"{DRIVE_OUT}/evaluation.json", binding, eval_records, scored, threshold)
report = evaluate(binding, eval_records, scored, threshold)
print(report["heldout_benign"]["pooled"]["fpr"], report["heldout_malicious"]["pooled"]["recall"])